In [ ]:
# ==============================================================================
# TRAINING SCRIPT: PHOBERT BASE -> FINE-TUNED V1
# Split: 70% Train - 15% Val - 15% Test
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH ---
DATA_FILE = '/content/drive/MyDrive/data MCQ 9K.jsonl'
BASE_MODEL_NAME = 'vinai/phobert-base'
OUTPUT_DIR = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model'
TEST_FILE_OUTPUT = '/content/drive/MyDrive/test_data_v1_new.jsonl' # Nơi lưu file Test để dành

BATCH_SIZE = 8
EPOCHS = 4
LEARNING_RATE = 2e-5
MAX_LEN = 256

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ============================================================
# 1. LOAD DATA
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading dataset: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path): return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items: process_item(obj, data)
                else:
                    obj = json.loads(line)
                    process_item(obj, data)
            except: continue

    print(f"   -> Loaded {len(data)} samples.")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer')

        if q and opts and ans:
            choices = []
            answer_idx = -1
            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]
                data_list.append({"question": str(q), "choices": choices, "label": answer_idx})
    except: pass

# ============================================================
# 2. SPLIT DATA (70 - 15 - 15)
# ============================================================
print("\n[INFO] Preparing Data & Splitting...")
all_data = load_data(DATA_FILE)
if not all_data: import sys; sys.exit()

# BƯỚC 1: Tách Test (15%) ra khỏi phần còn lại (85%)
# test_size=0.15 tương đương 15% tổng dữ liệu
train_val_data, test_data = train_test_split(all_data, test_size=0.15, random_state=42)

# BƯỚC 2: Tách Validation (15% TỔNG) từ phần còn lại
# Ta cần Val chiếm 15% tổng, tức là khoảng 17.6% của phần còn lại (0.15 / 0.85 ≈ 0.1765)
val_split_ratio = 0.15 / 0.85
train_data, val_data = train_test_split(train_val_data, test_size=val_split_ratio, random_state=42)

print(f"   📊 Tổng cộng: {len(all_data)}")
print(f"   🔹 Train (70%): {len(train_data)} câu")
print(f"   🔹 Val   (15%): {len(val_data)} câu")
print(f"   🔹 Test  (15%): {len(test_data)} câu")

# LƯU FILE TEST RA RIÊNG (ĐỂ DÀNH CHẤM SAU)
print(f"\n[INFO] Saving Test set to: {TEST_FILE_OUTPUT}")
with open(TEST_FILE_OUTPUT, 'w', encoding='utf-8') as f:
    for item in test_data:
        # Format lại để giống cấu trúc file gốc nhằm tái sử dụng code chấm thi
        save_obj = {
            "question": item['question'],
            "raw_options": {k: v for k, v in zip(['A','B','C','D'], item['choices'])},
            "raw_answer": ['A','B','C','D'][item['label']]
        }
        f.write(json.dumps(save_obj, ensure_ascii=False) + '\n')

# ============================================================
# 3. SETUP MODEL & TRAINING
# ============================================================
print(f"\n[INFO] Loading Base Model: {BASE_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=False)
model = AutoModelForMultipleChoice.from_pretrained(BASE_MODEL_NAME).to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer([item["question"]]*4, item["choices"],
                                max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

train_loader = DataLoader(QADataset(tokenizer, train_data), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_data), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# 4. TRAINING LOOP
# ============================================================
print("\n" + "="*40 + "\n🚀 START TRAINING\n" + "="*40)
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    # Validation
    model.eval()
    val_loss = 0; correct = 0; total = 0
    print("   -> Validating...")
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   📊 Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"   🔥 Saving Best Model to {OUTPUT_DIR}...")
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

print("\n🏆 TRAINING COMPLETE!")

Mounted at /content/drive
[INFO] Device: cuda

[INFO] Preparing Data & Splitting...
[INFO] Reading dataset: data MCQ 9K.jsonl
   -> Loaded 8946 samples.
   📊 Tổng cộng: 8946
   🔹 Train (70%): 6262 câu
   🔹 Val   (15%): 1342 câu
   🔹 Test  (15%): 1342 câu

[INFO] Saving Test set to: /content/drive/MyDrive/test_data_v1_new.jsonl

[INFO] Loading Base Model: vinai/phobert-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...

Some weights of RobertaForMultipleChoice were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]


🚀 START TRAINING

Epoch 1/4



Training:   4%|▎         | 29/783 [00:37<16:13,  1.29s/it, loss=1.4]


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI TỔNG QUÁT)
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (CHỈ CẦN SỬA Ở ĐÂY) ---

# 1. Đường dẫn Model cần chấm (Ví dụ: Trỏ vào V1)
CURRENT_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model'

# 2. File dữ liệu đề thi (Test Set)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai)
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                # Kiểm tra đủ trường thông tin
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys())

                    # Chỉ lấy những câu mà đáp án nằm trong options
                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Tokenize: [CLS] Question [SEP] Choice [SEP]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer (Cơ chế an toàn)
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        # 1. Thử load Tokenizer từ thư mục model
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'vinai/phobert-base'")
            tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

        # 2. Load Model Weights
        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại các câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": char_map[preds[i]] if preds[i] < 4 else "Unknown"
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()

[INFO] Device: cuda
[INFO] Reading data from: data MCQ 9K.jsonl
[INFO] Loaded 8946 valid samples.

[INFO] Loading model from: /content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V2
[INFO] Model loaded successfully.

[INFO] Starting Scoring Process...


Evaluating:  51%|█████     | 285/560 [04:05<03:56,  1.16it/s]


KeyboardInterrupt: 

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/generated_data_groq_v1.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [1]:
# ==============================================================================
# TRAIN PHOBERT V2: KẾT HỢP DỮ LIỆU GỐC & DỮ LIỆU SINH
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
drive.mount('/content/drive', force_remount=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị: {DEVICE}")

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# 1. Model V1 (Làm nền tảng)
OLD_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model'

# 2. Dữ liệu
FILE_ORIGINAL = '/content/drive/MyDrive/data MCQ 9K.jsonl'
# File bạn vừa sinh ra (dù tên là v3 nhưng dùng để train v2 cũng được)
FILE_GENERATED = '/content/drive/MyDrive/generated_data_v1.jsonl'

# 3. Nơi lưu V2
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V2'

# Tham số
BATCH_SIZE = 8
EPOCHS = 4        # Train kỹ hơn một chút
LR = 1e-5         # Learning rate nhỏ để fine-tune

if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

# ============================================================
# PHẦN 1: HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path, tag="Data"):
    print(f"📂 Đang đọc {tag}: {os.path.basename(file_path)}...")
    data = []

    if not os.path.exists(file_path):
        print(f"   ⚠️ Không tìm thấy file: {file_path}")
        return []

    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                # Xử lý cả JSON List và JSON Lines
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items:
                            if process_item(obj, data): count += 1
                else:
                    obj = json.loads(line)
                    if process_item(obj, data): count += 1
            except: continue

    print(f"   -> Lấy được: {count} câu.")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer') or obj.get('true_answer')

        if q and opts and ans:
            choices = []
            answer_idx = -1

            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                # Pad đủ 4 lựa chọn
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]

                data_list.append({
                    "question": str(q),
                    "choices": choices,
                    "label": answer_idx
                })
                return True
    except: pass
    return False

# ============================================================
# PHẦN 2: CHUẨN BỊ TRAIN
# ============================================================
print("\n" + "="*40)
print("🥣 TRỘN DỮ LIỆU CHO V2")
print("="*40)

# 1. Load dữ liệu
data_orig = load_data(FILE_ORIGINAL, "Gốc")
data_gen = load_data(FILE_GENERATED, "Sinh thêm")

# 2. Gộp lại
final_train_data = data_orig + data_gen
random.shuffle(final_train_data)

print(f"\n📚 TỔNG CỘNG: {len(final_train_data)} mẫu.")

if len(final_train_data) == 0:
    import sys; sys.exit("❌ Không có dữ liệu để train!")

# 3. Load Model V1
print(f"\n📥 Đang tải Model V1 từ: {OLD_MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH, use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained(OLD_MODEL_PATH).to(DEVICE)
    print("✅ Load V1 thành công!")
except:
    print("⚠️ Không tìm thấy V1. Đang tải PhoBERT Base...")
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained("vinai/phobert-base").to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer([item["question"]]*len(item["choices"]), item["choices"],
                                max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# Chia tập
train_set, val_set = train_test_split(final_train_data, test_size=0.1, random_state=42)
train_loader = DataLoader(QADataset(tokenizer, train_set, 256), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_set, 256), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# PHẦN 3: HUẤN LUYỆN V2
# ============================================================
print("\n" + "="*40)
print("🚀 BẮT ĐẦU HUẤN LUYỆN V2")
print("="*40)

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        # Reshape
        input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
        attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    # Validation
    model.eval()
    val_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
            attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   -> Val Loss: {avg_loss:.4f} | Val Accuracy: {acc:.4f}")

    # Lưu Model V2
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"🔥 Lưu V2 tốt nhất vào: {NEW_MODEL_OUTPUT_DIR}")
        model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🏆 V2 ĐÃ SẴN SÀNG!")

Mounted at /content/drive
🚀 Thiết bị: cuda

🥣 TRỘN DỮ LIỆU CHO V2
📂 Đang đọc Gốc: data MCQ 9K.jsonl...
   -> Lấy được: 8946 câu.
📂 Đang đọc Sinh thêm: generated_data_v1.jsonl...
   ⚠️ Không tìm thấy file: /content/drive/MyDrive/generated_data_v1.jsonl

📚 TỔNG CỘNG: 8946 mẫu.

📥 Đang tải Model V1 từ: /content/drive/MyDrive/PhoBert_FineTuned_QA_Model
⚠️ Không tìm thấy V1. Đang tải PhoBERT Base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...


🚀 BẮT ĐẦU HUẤN LUYỆN V2

Epoch 1/4


Training:   0%|          | 0/1007 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Training:   0%|          | 0/1007 [00:00<?, ?it/s]Exception ignored in: <generator object tqdm.__iter__ at 0x7db0b81432e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1196, in __iter__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1302, in close
    self.display(pos=0)
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1495, in display
    self.sp(self.__str__() if msg is None else msg)
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 459, in print_status
    fp_write('\r' + s + (' ' * max(last_len[0] - len_s, 0)))
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 453, in fp_write
    fp_flush()
  File "/usr/local/lib/python3.12/dist-packages/tqdm/utils.py", line 196, in inner
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 488, in flush
    if not

KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI TỔNG QUÁT)
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (CHỈ CẦN SỬA Ở ĐÂY) ---

# 1. Đường dẫn Model cần chấm (Ví dụ: Trỏ vào V1)
CURRENT_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V2'

# 2. File dữ liệu đề thi (Test Set)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai)
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model_V2.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                # Kiểm tra đủ trường thông tin
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys())

                    # Chỉ lấy những câu mà đáp án nằm trong options
                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Tokenize: [CLS] Question [SEP] Choice [SEP]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer (Cơ chế an toàn)
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        # 1. Thử load Tokenizer từ thư mục model
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'vinai/phobert-base'")
            tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

        # 2. Load Model Weights
        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại các câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": char_map[preds[i]] if preds[i] < 4 else "Unknown"
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model_V2.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/Generated_data_distil/generated_data_groq_v2.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [ ]:
# ==============================================================================
# TRAIN PHOBERT V2: KẾT HỢP DỮ LIỆU GỐC & DỮ LIỆU SINH
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
drive.mount('/content/drive', force_remount=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị: {DEVICE}")

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# 1. Model V1 (Làm nền tảng)
OLD_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V2'

# 2. Dữ liệu
FILE_ORIGINAL = '/content/drive/MyDrive/data MCQ 9K.jsonl'
# File bạn vừa sinh ra (dù tên là v3 nhưng dùng để train v2 cũng được)
FILE_GENERATED = '/content/drive/MyDrive/generated_data_v2.jsonl'

# 3. Nơi lưu V2
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V3'

# Tham số
BATCH_SIZE = 8
EPOCHS = 4        # Train kỹ hơn một chút
LR = 1e-5         # Learning rate nhỏ để fine-tune

if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

# ============================================================
# PHẦN 1: HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path, tag="Data"):
    print(f"📂 Đang đọc {tag}: {os.path.basename(file_path)}...")
    data = []

    if not os.path.exists(file_path):
        print(f"   ⚠️ Không tìm thấy file: {file_path}")
        return []

    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                # Xử lý cả JSON List và JSON Lines
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items:
                            if process_item(obj, data): count += 1
                else:
                    obj = json.loads(line)
                    if process_item(obj, data): count += 1
            except: continue

    print(f"   -> Lấy được: {count} câu.")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer') or obj.get('true_answer')

        if q and opts and ans:
            choices = []
            answer_idx = -1

            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                # Pad đủ 4 lựa chọn
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]

                data_list.append({
                    "question": str(q),
                    "choices": choices,
                    "label": answer_idx
                })
                return True
    except: pass
    return False

# ============================================================
# PHẦN 2: CHUẨN BỊ TRAIN
# ============================================================
print("\n" + "="*40)
print("🥣 TRỘN DỮ LIỆU CHO V2")
print("="*40)

# 1. Load dữ liệu
data_orig = load_data(FILE_ORIGINAL, "Gốc")
data_gen = load_data(FILE_GENERATED, "Sinh thêm")

# 2. Gộp lại
final_train_data = data_orig + data_gen
random.shuffle(final_train_data)

print(f"\n📚 TỔNG CỘNG: {len(final_train_data)} mẫu.")

if len(final_train_data) == 0:
    import sys; sys.exit("❌ Không có dữ liệu để train!")

# 3. Load Model V1
print(f"\n📥 Đang tải Model V1 từ: {OLD_MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH, use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained(OLD_MODEL_PATH).to(DEVICE)
    print("✅ Load V1 thành công!")
except:
    print("⚠️ Không tìm thấy V1. Đang tải PhoBERT Base...")
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained("vinai/phobert-base").to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer([item["question"]]*len(item["choices"]), item["choices"],
                                max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# Chia tập
train_set, val_set = train_test_split(final_train_data, test_size=0.1, random_state=42)
train_loader = DataLoader(QADataset(tokenizer, train_set, 256), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_set, 256), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# PHẦN 3: HUẤN LUYỆN V3
# ============================================================
print("\n" + "="*40)
print("🚀 BẮT ĐẦU HUẤN LUYỆN V2")
print("="*40)

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        # Reshape
        input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
        attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    # Validation
    model.eval()
    val_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
            attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   -> Val Loss: {avg_loss:.4f} | Val Accuracy: {acc:.4f}")

    # Lưu Model V2
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"🔥 Lưu V3 tốt nhất vào: {NEW_MODEL_OUTPUT_DIR}")
        model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🏆 V3 ĐÃ SẴN SÀNG!")

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI TỔNG QUÁT)
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (CHỈ CẦN SỬA Ở ĐÂY) ---

# 1. Đường dẫn Model cần chấm (Ví dụ: Trỏ vào V1)
CURRENT_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V3'

# 2. File dữ liệu đề thi (Test Set)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai)
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model_V3.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                # Kiểm tra đủ trường thông tin
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys())

                    # Chỉ lấy những câu mà đáp án nằm trong options
                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Tokenize: [CLS] Question [SEP] Choice [SEP]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer (Cơ chế an toàn)
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        # 1. Thử load Tokenizer từ thư mục model
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'vinai/phobert-base'")
            tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

        # 2. Load Model Weights
        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại các câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": char_map[preds[i]] if preds[i] < 4 else "Unknown"
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model_V3.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/generated_data_groq_v3.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [ ]:
# ==============================================================================
# TRAIN PHOBERT V2: KẾT HỢP DỮ LIỆU GỐC & DỮ LIỆU SINH
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
drive.mount('/content/drive', force_remount=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị: {DEVICE}")

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# 1. Model V1 (Làm nền tảng)
OLD_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V3'

# 2. Dữ liệu
FILE_ORIGINAL = '/content/drive/MyDrive/data MCQ 9K.jsonl'
# File bạn vừa sinh ra (dù tên là v3 nhưng dùng để train v2 cũng được)
FILE_GENERATED = '/content/drive/MyDrive/generated_data_v3.jsonl'

# 3. Nơi lưu V2
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V4'

# Tham số
BATCH_SIZE = 8
EPOCHS = 4        # Train kỹ hơn một chút
LR = 1e-5         # Learning rate nhỏ để fine-tune

if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

# ============================================================
# PHẦN 1: HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path, tag="Data"):
    print(f"📂 Đang đọc {tag}: {os.path.basename(file_path)}...")
    data = []

    if not os.path.exists(file_path):
        print(f"   ⚠️ Không tìm thấy file: {file_path}")
        return []

    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                # Xử lý cả JSON List và JSON Lines
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items:
                            if process_item(obj, data): count += 1
                else:
                    obj = json.loads(line)
                    if process_item(obj, data): count += 1
            except: continue

    print(f"   -> Lấy được: {count} câu.")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer') or obj.get('true_answer')

        if q and opts and ans:
            choices = []
            answer_idx = -1

            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                # Pad đủ 4 lựa chọn
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]

                data_list.append({
                    "question": str(q),
                    "choices": choices,
                    "label": answer_idx
                })
                return True
    except: pass
    return False

# ============================================================
# PHẦN 2: CHUẨN BỊ TRAIN
# ============================================================
print("\n" + "="*40)
print("🥣 TRỘN DỮ LIỆU CHO V2")
print("="*40)

# 1. Load dữ liệu
data_orig = load_data(FILE_ORIGINAL, "Gốc")
data_gen = load_data(FILE_GENERATED, "Sinh thêm")

# 2. Gộp lại
final_train_data = data_orig + data_gen
random.shuffle(final_train_data)

print(f"\n📚 TỔNG CỘNG: {len(final_train_data)} mẫu.")

if len(final_train_data) == 0:
    import sys; sys.exit("❌ Không có dữ liệu để train!")

# 3. Load Model V1
print(f"\n📥 Đang tải Model V1 từ: {OLD_MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH, use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained(OLD_MODEL_PATH).to(DEVICE)
    print("✅ Load V1 thành công!")
except:
    print("⚠️ Không tìm thấy V1. Đang tải PhoBERT Base...")
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained("vinai/phobert-base").to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer([item["question"]]*len(item["choices"]), item["choices"],
                                max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# Chia tập
train_set, val_set = train_test_split(final_train_data, test_size=0.1, random_state=42)
train_loader = DataLoader(QADataset(tokenizer, train_set, 256), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_set, 256), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# PHẦN 3: HUẤN LUYỆN V3
# ============================================================
print("\n" + "="*40)
print("🚀 BẮT ĐẦU HUẤN LUYỆN V2")
print("="*40)

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        # Reshape
        input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
        attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    # Validation
    model.eval()
    val_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
            attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   -> Val Loss: {avg_loss:.4f} | Val Accuracy: {acc:.4f}")

    # Lưu Model V2
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"🔥 Lưu V4 tốt nhất vào: {NEW_MODEL_OUTPUT_DIR}")
        model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🏆 V4 ĐÃ SẴN SÀNG!")

In [ ]:
# ==============================================================================
# GENERIC EVALUATION SCRIPT (CHẤM THI TỔNG QUÁT)
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Device: {DEVICE}")

# --- CẤU HÌNH (CHỈ CẦN SỬA Ở ĐÂY) ---

# 1. Đường dẫn Model cần chấm (Ví dụ: Trỏ vào V1)
CURRENT_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V4'

# 2. File dữ liệu đề thi (Test Set)
TEST_DATA_PATH = '/content/drive/MyDrive/data MCQ 9K.jsonl'

# 3. Nơi lưu kết quả (File chứa các câu làm sai)
OUTPUT_WRONG_ANSWERS = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model_V4.jsonl'

# Tham số khác
BATCH_SIZE = 16
MAX_LEN = 256

# ============================================================
# 1. HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path):
    print(f"[INFO] Reading data from: {os.path.basename(file_path)}")
    data = []
    if not os.path.exists(file_path):
        print(f"[ERROR] File not found: {file_path}")
        return []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                # Kiểm tra đủ trường thông tin
                if 'question' in obj and 'options' in obj and 'answer' in obj:
                    opts = obj['options']
                    sorted_k = sorted(opts.keys())

                    # Chỉ lấy những câu mà đáp án nằm trong options
                    if obj['answer'] in sorted_k:
                        data.append({
                            "question": obj['question'],
                            "choices": [opts[k] for k in sorted_k],
                            "label": sorted_k.index(obj['answer']),
                            "raw_options": opts,
                            "raw_answer": obj['answer']
                        })
            except: continue

    print(f"[INFO] Loaded {len(data)} valid samples.")
    return data

# ============================================================
# 2. DATASET CLASS
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Tokenize: [CLS] Question [SEP] Choice [SEP]
        inputs = self.tokenizer(
            [item["question"]] * len(item["choices"]),
            item["choices"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# ============================================================
# 3. HÀM CHẤM THI (MAIN EVALUATION)
# ============================================================
def run_evaluation():
    # Load Data
    all_data = load_data(TEST_DATA_PATH)
    if not all_data: return

    # Load Model & Tokenizer (Cơ chế an toàn)
    print(f"\n[INFO] Loading model from: {CURRENT_MODEL_PATH}")
    try:
        # 1. Thử load Tokenizer từ thư mục model
        try:
            tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL_PATH)
        except:
            print("[WARN] Local tokenizer not found. Using fallback: 'vinai/phobert-base'")
            tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

        # 2. Load Model Weights
        model = AutoModelForMultipleChoice.from_pretrained(CURRENT_MODEL_PATH).to(DEVICE)
        model.eval()
        print("[INFO] Model loaded successfully.")

    except Exception as e:
        print(f"[ERROR] Critical failure loading model: {e}")
        return

    # Prepare Loader
    dataset = EvalDataset(tokenizer, all_data, max_len=MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n[INFO] Starting Scoring Process...")

    all_preds = []
    all_labels = []
    wrong_cases = []
    batch_start_idx = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            targets = labels.cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(targets)

            # Ghi lại các câu sai
            for i in range(len(preds)):
                if preds[i] != targets[i]:
                    original_item = all_data[batch_start_idx + i]
                    char_map = ['A', 'B', 'C', 'D']
                    wrong_cases.append({
                        "question": original_item['question'],
                        "options": original_item['raw_options'],
                        "correct_answer": original_item['raw_answer'],
                        "model_prediction": char_map[preds[i]] if preds[i] < 4 else "Unknown"
                    })

            batch_start_idx += len(preds)

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)

    print("\n" + "="*40)
    print(f"REPORT RESULTS")
    print("="*40)
    print(f"Model Path:      {os.path.basename(CURRENT_MODEL_PATH)}")
    print(f"Total Questions: {len(all_data)}")
    print(f"Correct:         {len(all_data) - len(wrong_cases)}")
    print(f"Wrong:           {len(wrong_cases)}")
    print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=['A', 'B', 'C', 'D'], zero_division=0))

    # Save Wrong Answers
    if wrong_cases:
        with open(OUTPUT_WRONG_ANSWERS, 'w', encoding='utf-8') as f:
            for item in wrong_cases:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"\n[SAVE] Saved {len(wrong_cases)} wrong answers to: {OUTPUT_WRONG_ANSWERS}")
    else:
        print("\n[INFO] Perfect Score! No wrong answers file generated.")

if __name__ == "__main__":
    run_evaluation()

In [ ]:

# ==============================================================================

# TOOL SINH DỮ LIỆU GROQ (PHIÊN BẢN TỐC ĐỘ CAO - 8B INSTANT)

# ==============================================================================



!pip install -q groq tqdm



import os

import json

import time

from groq import Groq, RateLimitError, APIError, BadRequestError

from google.colab import drive

from tqdm import tqdm



# 1. Kết nối Drive

if not os.path.exists('/content/drive'):

    drive.mount('/content/drive', force_remount=True)



# --- CẤU HÌNH ---

GROQ_API_KEY = "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"



# 🔥 QUAN TRỌNG: CHUYỂN SANG 8B ĐỂ CÓ QUOTA CAO HƠN

GROQ_MODEL = "qwen/qwen3-32b"



INPUT_FILE = '/content/drive/MyDrive/Benchmark_Results/wrong_answers_PhoBert_FineTuned_QA_Model_V4.jsonl'

OUTPUT_FILE = '/content/drive/MyDrive/generated_data_groq_v4.jsonl'



# 🔥 GIẢM BATCH XUỐNG 1 ĐỂ AN TOÀN TUYỆT ĐỐI VỚI RATE LIMIT

BATCH_SIZE = 5

NUM_VARIANTS = 4



client = Groq(api_key=GROQ_API_KEY)



# ============================================================

# HÀM GỌI API

# ============================================================

def get_groq_response(batch_content):



    system_prompt = "You are an Expert Exam Creator. You output JSON only."



    # Prompt được tối ưu cho 8B

    user_prompt = f"""

    Task: Create exactly {NUM_VARIANTS} NEW multiple-choice variants for the input question.



    INPUT:

    {json.dumps(batch_content, ensure_ascii=False)}



    OUTPUT JSON FORMAT:

    {{

      "questions": [

        {{ "question": "...", "options": {{ "A": "...", "B": "...", "C": "...", "D": "..." }}, "answer": "A" }},

        ...

      ]

    }}

    STRICTLY JSON. NO EXPLANATION.

    """



    max_retries = 10

    for attempt in range(max_retries):

        try:

            chat_completion = client.chat.completions.create(

                messages=[

                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": user_prompt}

                ],

                model=GROQ_MODEL,

                response_format={"type": "json_object"},

                temperature=0.6,

                max_tokens=2048 # Giới hạn output để tiết kiệm TPM

            )

            return chat_completion.choices[0].message.content



        except RateLimitError:

            wait_time = 10 + (attempt * 5) # Chờ tăng dần: 10s, 15s, 20s...

            print(f"\r   ⏳ Hết Quota 8B. Đang nghỉ {wait_time}s... (Lần {attempt+1})", end="")

            time.sleep(wait_time)



        except (APIError, BadRequestError) as e:

            print(f"\n   ⚠️ Lỗi API: {e}. Thử lại sau 2s...")

            time.sleep(2)

        except Exception as e:

            print(f"\n   ⚠️ Lỗi lạ: {e}")

            time.sleep(2)



    return None



# ============================================================

# MAIN LOOP

# ============================================================

def main():

    print(f"🚀 BẮT ĐẦU VỚI MODEL: {GROQ_MODEL} (Tốc độ cao)")



    if not os.path.exists(INPUT_FILE):

        print(f"❌ Lỗi: Không tìm thấy file {INPUT_FILE}")

        return



    input_data = []

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:

        for line in f:

            try: input_data.append(json.loads(line))

            except: continue



    total_input = len(input_data)

    print(f"📚 Tổng số câu sai cần xử lý: {total_input}")



    # Resume Logic

    start_idx = 0

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, 'r') as f:

            existing_lines = sum(1 for _ in f)



        # Tính toán lại vị trí resume

        processed_items = existing_lines // NUM_VARIANTS

        start_idx = processed_items



        # Căn chỉnh batch

        start_idx = (start_idx // BATCH_SIZE) * BATCH_SIZE

        print(f"🔄 File cũ có {existing_lines} dòng. Tiếp tục từ câu gốc thứ {start_idx}...")



    print("\n⚡ Đang chạy...")



    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:



        # Cắt danh sách input

        remaining_data = input_data[start_idx:]

        batches = [remaining_data[i:i + BATCH_SIZE] for i in range(0, len(remaining_data), BATCH_SIZE)]



        for batch in tqdm(batches, desc="Tiến độ"):



            # Chuẩn bị Input

            batch_content = []

            for item in batch:

                q = item.get('question') or item.get('q')

                opts = item.get('options') or item.get('raw_options') or item.get('opts')

                ans = item.get('answer') or item.get('raw_answer') or item.get('correct_answer')

                if q and opts and ans:

                    batch_content.append({"q": q, "opts": opts, "ans": ans})



            if not batch_content: continue



            # Gọi Groq

            json_str = get_groq_response(batch_content)



            if json_str:

                try:

                    data = json.loads(json_str)

                    new_qs = []



                    if isinstance(data, dict):

                        # Ưu tiên tìm key "questions"

                        if "questions" in data and isinstance(data["questions"], list):

                            new_qs = data["questions"]

                        else:

                            # Tìm bất kỳ key nào chứa list

                            for key in data:

                                if isinstance(data[key], list):

                                    new_qs = data[key]

                                    break

                    elif isinstance(data, list):

                        new_qs = data



                    for q in new_qs:

                        if 'question' in q and 'options' in q:

                            f_out.write(json.dumps(q, ensure_ascii=False) + "\n")



                    f_out.flush()

                    os.fsync(f_out.fileno())



                except json.JSONDecodeError:

                    print("   ❌ Lỗi Parse JSON")



            # Model 8B rất nhanh, nghỉ cực ngắn là đủ

            time.sleep(0.5)



    print("\n" + "="*50)

    print("✅ HOÀN TẤT QUÁ TRÌNH!")

    print(f"📂 File kết quả: {OUTPUT_FILE}")



if __name__ == "__main__":

  main()

In [ ]:
# ==============================================================================
# TRAIN PHOBERT V2: KẾT HỢP DỮ LIỆU GỐC & DỮ LIỆU SINH
# ==============================================================================

# 1. Cài đặt thư viện
!pip install -q transformers torch scikit-learn pandas accelerate

import os
import json
import torch
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

# 2. Kết nối Drive
drive.mount('/content/drive', force_remount=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị: {DEVICE}")

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# 1. Model V1 (Làm nền tảng)
OLD_MODEL_PATH = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V4'

# 2. Dữ liệu
FILE_ORIGINAL = '/content/drive/MyDrive/data MCQ 9K.jsonl'
# File bạn vừa sinh ra (dù tên là v3 nhưng dùng để train v2 cũng được)
FILE_GENERATED = '/content/drive/MyDrive/generated_data_v4.jsonl'

# 3. Nơi lưu V2
NEW_MODEL_OUTPUT_DIR = '/content/drive/MyDrive/PhoBert_FineTuned_QA_Model_V5'

# Tham số
BATCH_SIZE = 8
EPOCHS = 4        # Train kỹ hơn một chút
LR = 1e-5         # Learning rate nhỏ để fine-tune

if not os.path.exists(NEW_MODEL_OUTPUT_DIR):
    os.makedirs(NEW_MODEL_OUTPUT_DIR)

# ============================================================
# PHẦN 1: HÀM ĐỌC DỮ LIỆU
# ============================================================
def load_data(file_path, tag="Data"):
    print(f"📂 Đang đọc {tag}: {os.path.basename(file_path)}...")
    data = []

    if not os.path.exists(file_path):
        print(f"   ⚠️ Không tìm thấy file: {file_path}")
        return []

    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                # Xử lý cả JSON List và JSON Lines
                if line.strip().startswith('['):
                    items = json.loads(line)
                    if isinstance(items, list):
                        for obj in items:
                            if process_item(obj, data): count += 1
                else:
                    obj = json.loads(line)
                    if process_item(obj, data): count += 1
            except: continue

    print(f"   -> Lấy được: {count} câu.")
    return data

def process_item(obj, data_list):
    try:
        q = obj.get('question')
        opts = obj.get('options') or obj.get('raw_options')
        ans = obj.get('answer') or obj.get('raw_answer') or obj.get('correct_answer') or obj.get('true_answer')

        if q and opts and ans:
            choices = []
            answer_idx = -1

            if isinstance(opts, dict):
                sorted_k = sorted(opts.keys())
                choices = [str(opts[k]) for k in sorted_k]
                if ans in sorted_k: answer_idx = sorted_k.index(ans)
            elif isinstance(opts, list):
                choices = [str(o) for o in opts]
                if isinstance(ans, str) and len(ans) == 1:
                    char_map = {'A':0, 'B':1, 'C':2, 'D':3}
                    answer_idx = char_map.get(ans.upper(), -1)

            if len(choices) >= 2 and answer_idx != -1 and answer_idx < 4:
                # Pad đủ 4 lựa chọn
                while len(choices) < 4: choices.append("-")
                choices = choices[:4]

                data_list.append({
                    "question": str(q),
                    "choices": choices,
                    "label": answer_idx
                })
                return True
    except: pass
    return False

# ============================================================
# PHẦN 2: CHUẨN BỊ TRAIN
# ============================================================
print("\n" + "="*40)
print("🥣 TRỘN DỮ LIỆU CHO V2")
print("="*40)

# 1. Load dữ liệu
data_orig = load_data(FILE_ORIGINAL, "Gốc")
data_gen = load_data(FILE_GENERATED, "Sinh thêm")

# 2. Gộp lại
final_train_data = data_orig + data_gen
random.shuffle(final_train_data)

print(f"\n📚 TỔNG CỘNG: {len(final_train_data)} mẫu.")

if len(final_train_data) == 0:
    import sys; sys.exit("❌ Không có dữ liệu để train!")

# 3. Load Model V1
print(f"\n📥 Đang tải Model V1 từ: {OLD_MODEL_PATH}")
try:
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH, use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained(OLD_MODEL_PATH).to(DEVICE)
    print("✅ Load V1 thành công!")
except:
    print("⚠️ Không tìm thấy V1. Đang tải PhoBERT Base...")
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
    model = AutoModelForMultipleChoice.from_pretrained("vinai/phobert-base").to(DEVICE)

class QADataset(Dataset):
    def __init__(self, tokenizer, data_list, max_len=256):
        self.tokenizer = tokenizer
        self.data = data_list
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer([item["question"]]*len(item["choices"]), item["choices"],
                                max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(item["label"], dtype=torch.long)
        }

# Chia tập
train_set, val_set = train_test_split(final_train_data, test_size=0.1, random_state=42)
train_loader = DataLoader(QADataset(tokenizer, train_set, 256), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(QADataset(tokenizer, val_set, 256), batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * EPOCHS)

# ============================================================
# PHẦN 3: HUẤN LUYỆN V3
# ============================================================
print("\n" + "="*40)
print("🚀 BẮT ĐẦU HUẤN LUYỆN V2")
print("="*40)

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    prog_bar = tqdm(train_loader, desc="Training")
    total_loss = 0

    for batch in prog_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        # Reshape
        input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
        attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        prog_bar.set_postfix({'loss': loss.item()})

    # Validation
    model.eval()
    val_loss = 0; correct = 0; total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            input_ids = input_ids.view(-1, 4, input_ids.shape[-1])
            attention_mask = attention_mask.view(-1, 4, attention_mask.shape[-1])

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    avg_loss = val_loss / len(val_loader)
    acc = correct / total
    print(f"   -> Val Loss: {avg_loss:.4f} | Val Accuracy: {acc:.4f}")

    # Lưu Model V2
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"🔥 Lưu V5 tốt nhất vào: {NEW_MODEL_OUTPUT_DIR}")
        model.save_pretrained(NEW_MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(NEW_MODEL_OUTPUT_DIR)

print("\n🏆 V5 ĐÃ SẴN SÀNG!")